# KDD Cup 2015 MOOC dropout - feature engineering, FL, and XAI

This notebook reproduces the KDD Cup 2015 experiment from raw competition CSVs. It aggregates enrollment activity, temporal engagement, active days 1-30, and prior course outcomes, then runs the centralized network, FedAvg, FedProx, and local explanations.

Sources: [official Biendata competition](https://www.biendata.xyz/competition/kddcup2015/), [ACM KDD challenge description](https://www.kdd.org/kdd2015/calls.html), and the [Kaggle mirror used by the cited baseline](https://www.kaggle.com/datasets/sst2023/kdd-cup-2015). Data are not redistributed here.

Put the raw files below `data/kddcup2015/raw/`, or set `AIED_KDD_KAGGLE_DOWNLOAD=1`. Set `AIED_PAPER_MODE=1` for the full 100-client, 50-round, 10-repetition experiment.

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

DEFAULT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPO_ROOT = Path(os.environ.get("AIED_REPO_ROOT", DEFAULT_ROOT)).resolve()
DATA_DIR = REPO_ROOT / "data" / "kddcup2015"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_CSV = PROCESSED_DIR / "kdd_cup_2015_features.csv"
RESULTS_DIR = REPO_ROOT / "results" / "kddcup2015"
PAPER_MODE = os.environ.get("AIED_PAPER_MODE", "0") == "1"
RUN_XAI = os.environ.get("AIED_RUN_XAI", "0") == "1"
REBUILD_FEATURES = os.environ.get("AIED_REBUILD_FEATURES", "0") == "1"
KAGGLE_DOWNLOAD = os.environ.get("AIED_KDD_KAGGLE_DOWNLOAD", "0") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "repeats": 10 if PAPER_MODE else 1,
    "central_epochs": 100 if PAPER_MODE else 2,
    "clients": 100 if PAPER_MODE else 3,
    "rounds": 50 if PAPER_MODE else 2,
    "local_epochs": 2 if PAPER_MODE else 1,
    "batch_size": 64,
    "lr": 0.02,
    "mu": 0.01,
    "fast_max_rows": None if PAPER_MODE else 5000,
}
print({"paper_mode": PAPER_MODE, "device": str(DEVICE), **CONFIG})

## 1. Locate raw files and engineer features

The Kaggle mirror may keep train and test files in separate subfolders; the recursive locator handles that layout. Feature generation can require several GB of RAM because the raw data contain roughly 13.5 million log events.

In [ ]:
REQUIRED = [
    "date.csv", "enrollment_train.csv", "enrollment_test.csv", "log_train.csv", "log_test.csv",
    "truth_train.csv", "truth_test.csv",
]

def locate_kdd_files(root):
    found = {}
    for name in REQUIRED:
        matches = list(Path(root).rglob(name))
        if not matches:
            raise FileNotFoundError(
                f"Missing {name} below {root}. Download the KDD Cup 2015 Kaggle mirror described above."
            )
        found[name] = matches[0]
    return found


def get_raw_root():
    if KAGGLE_DOWNLOAD:
        import kagglehub
        downloaded = kagglehub.dataset_download("sst2023/kdd-cup-2015")
        return Path(downloaded)
    return RAW_DIR


def engineer_kdd_features():
    files = locate_kdd_files(get_raw_root())
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    logs = pd.concat(
        [
            pd.read_csv(files["log_train.csv"], usecols=["enrollment_id", "time", "source", "event"]),
            pd.read_csv(files["log_test.csv"], usecols=["enrollment_id", "time", "source", "event"]),
        ],
        ignore_index=True,
    )
    logs["time"] = pd.to_datetime(logs["time"])
    logs["day"] = logs["time"].dt.normalize()
    enrollments = pd.concat(
        [pd.read_csv(files["enrollment_train.csv"]), pd.read_csv(files["enrollment_test.csv"])],
        ignore_index=True,
    )
    course_dates = pd.read_csv(files["date.csv"])
    course_dates["from"] = pd.to_datetime(course_dates["from"])
    course_dates["to"] = pd.to_datetime(course_dates["to"])
    labels = pd.concat(
        [
            pd.read_csv(files["truth_train.csv"], names=["enroll_id", "dropout"]),
            pd.read_csv(files["truth_test.csv"], names=["enroll_id", "dropout"]),
        ],
        ignore_index=True,
    )

    basic = logs.groupby("enrollment_id").agg(
        total_count=("event", "size"),
        days_active_count=("day", "nunique"),
        first_day=("day", "min"),
        last_day=("day", "max"),
    )
    basic["date_range"] = (basic.pop("last_day") - basic.pop("first_day")).dt.days
    source_counts = logs.groupby(["enrollment_id", "source"]).size().unstack(fill_value=0)
    basic["server_count"] = source_counts.get("server", 0)
    basic["browser_count"] = source_counts.get("browser", 0)

    event_source = logs.groupby(["enrollment_id", "event", "source"]).size().unstack(["event", "source"], fill_value=0)
    combinations = {
        "navigate_s": ("navigate", "server"), "access_s": ("access", "server"),
        "access_b": ("access", "browser"), "problem_s": ("problem", "server"),
        "problem_b": ("problem", "browser"), "video_b": ("video", "browser"),
        "page_close_b": ("page_close", "browser"), "wiki_s": ("wiki", "server"),
        "discussion_s": ("discussion", "server"),
    }
    for feature, pair in combinations.items():
        basic[feature] = event_source[pair] if pair in event_source.columns else 0
    basic = basic.reset_index().rename(columns={"enrollment_id": "enroll_id"})

    enrollment_dates = enrollments.merge(course_dates, on="course_id", how="left")
    starts = enrollment_dates.set_index("enrollment_id")["from"]
    logs["active_day"] = (logs["day"] - logs["enrollment_id"].map(starts)).dt.days
    active = logs.loc[logs["active_day"].between(0, 29), ["enrollment_id", "active_day"]].drop_duplicates()
    active["value"] = 1
    active = active.pivot(index="enrollment_id", columns="active_day", values="value").fillna(0)
    active = active.reindex(columns=range(30), fill_value=0)
    active.columns = [f"active_day_{i + 1}" for i in range(30)]
    active = active.reset_index().rename(columns={"enrollment_id": "enroll_id"})

    history = enrollment_dates[["enrollment_id", "username", "to"]].merge(
        labels, left_on="enrollment_id", right_on="enroll_id", how="inner"
    )
    previous = {}
    for _, group in history.groupby("username", sort=False):
        dates = group["to"].to_numpy(dtype="datetime64[D]")
        outcomes = group["dropout"].to_numpy(dtype=np.int64)
        ids = group["enroll_id"].to_numpy()
        for current_id, current_date in zip(ids, dates):
            eligible = dates < (current_date - np.timedelta64(10, "D"))
            previous[current_id] = (
                int(np.sum(eligible & (outcomes == 0))),
                int(np.sum(eligible & (outcomes == 1))),
            )
    previous = pd.DataFrame.from_dict(
        previous, orient="index", columns=["prev_complete", "prev_dropout"]
    ).rename_axis("enroll_id").reset_index()

    features = labels.merge(basic, on="enroll_id", how="left")
    features = features.merge(active, on="enroll_id", how="left")
    features = features.merge(previous, on="enroll_id", how="left")
    features = features.fillna(0)
    ordered = ["enroll_id", "dropout"] + [c for c in features.columns if c not in ["enroll_id", "dropout"]]
    features = features[ordered]
    features.to_csv(FEATURES_CSV, index=False)
    print("Engineered shape:", features.shape)
    return features


if REBUILD_FEATURES or not FEATURES_CSV.exists():
    kdd_features = engineer_kdd_features()
else:
    kdd_features = pd.read_csv(FEATURES_CSV)
display(kdd_features.head())
print(kdd_features.shape, kdd_features["dropout"].value_counts(normalize=True).round(3).to_dict())

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


class DropoutMLP(nn.Module):
    """Two-hidden-layer network used in the paper (30 and 10 units)."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 30),
            nn.ReLU(),
            nn.Linear(30, 10),
            nn.ReLU(),
            nn.Linear(10, 2),
        )

    def forward(self, x):
        return self.layers(x)


def inverse_frequency_weights(y, device):
    counts = np.bincount(np.asarray(y, dtype=np.int64), minlength=2)
    if np.any(counts == 0):
        raise ValueError(f"Both classes must be present in the training split; got {counts.tolist()}")
    weights = len(y) / (2.0 * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def make_loader(X, y, batch_size, shuffle, seed):
    dataset = TensorDataset(
        torch.as_tensor(X, dtype=torch.float32),
        torch.as_tensor(y, dtype=torch.long),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


def train_central(X, y, epochs, batch_size, lr, seed, device):
    set_seed(seed)
    model = DropoutMLP(X.shape[1]).to(device)
    criterion = nn.CrossEntropyLoss(weight=inverse_frequency_weights(y, device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = make_loader(X, y, batch_size, True, seed + 1)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


def partition_clients(X, y, n_clients, seed):
    if n_clients > len(y):
        raise ValueError("The number of clients cannot exceed the number of training rows.")
    rng = np.random.default_rng(seed)
    partitions = np.array_split(rng.permutation(len(y)), n_clients)
    return [(X[idx], y[idx]) for idx in partitions]


def train_federated(X, y, n_clients, rounds, local_epochs, batch_size, lr, mu, seed, device):
    """FedAvg when mu=0, and FedProx when mu>0."""
    set_seed(seed)
    global_model = DropoutMLP(X.shape[1]).to(device)
    clients = partition_clients(X, y, n_clients, seed + 1)
    class_weights = inverse_frequency_weights(y, device)
    client_sizes = np.asarray([len(cy) for _, cy in clients], dtype=np.float64)
    aggregation_weights = client_sizes / client_sizes.sum()

    for round_idx in range(rounds):
        local_states = []
        for client_idx, (client_X, client_y) in enumerate(clients):
            local_model = DropoutMLP(X.shape[1]).to(device)
            local_model.load_state_dict(global_model.state_dict())
            global_reference = [p.detach().clone() for p in global_model.parameters()]
            criterion = nn.CrossEntropyLoss(weight=class_weights)
            optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)
            loader_seed = seed + 10_000 * round_idx + client_idx
            loader = make_loader(client_X, client_y, batch_size, True, loader_seed)

            for _ in range(local_epochs):
                local_model.train()
                for xb, yb in loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad(set_to_none=True)
                    loss = criterion(local_model(xb), yb)
                    if mu > 0:
                        proximal = sum(
                            torch.sum((local - reference) ** 2)
                            for local, reference in zip(local_model.parameters(), global_reference)
                        )
                        loss = loss + (mu / 2.0) * proximal
                    loss.backward()
                    optimizer.step()
            local_states.append({k: v.detach().clone() for k, v in local_model.state_dict().items()})

        aggregated = {}
        for name in global_model.state_dict():
            aggregated[name] = sum(
                float(weight) * state[name]
                for weight, state in zip(aggregation_weights, local_states)
            )
        global_model.load_state_dict(aggregated)
    return global_model


def evaluate(model, X, y, f1_average, device):
    model.eval()
    with torch.no_grad():
        logits = model(torch.as_tensor(X, dtype=torch.float32, device=device))
        probabilities = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        predictions = logits.argmax(dim=1).cpu().numpy()
    return {
        "accuracy": accuracy_score(y, predictions),
        "f1": f1_score(y, predictions, average=f1_average, zero_division=0),
        "auc": roc_auc_score(y, probabilities),
    }


def run_three_methods(X_train, y_train, X_test, y_test, config, f1_average, seed, device):
    central = train_central(
        X_train, y_train, config["central_epochs"], config["batch_size"],
        config["lr"], seed + 100, device,
    )
    fedavg = train_federated(
        X_train, y_train, config["clients"], config["rounds"],
        config["local_epochs"], config["batch_size"], config["lr"],
        0.0, seed + 200, device,
    )
    fedprox = train_federated(
        X_train, y_train, config["clients"], config["rounds"],
        config["local_epochs"], config["batch_size"], config["lr"],
        config["mu"], seed + 200, device,
    )
    models = {"Central": central, "FedAvg": fedavg, "FedProx": fedprox}
    metrics = {name: evaluate(model, X_test, y_test, f1_average, device) for name, model in models.items()}
    return metrics, models

## 2. Run centralized, FedAvg, and FedProx experiments

In [ ]:
X_frame = kdd_features.drop(columns=["enroll_id", "dropout"])
X = X_frame.to_numpy(dtype=np.float32)
y = kdd_features["dropout"].to_numpy(dtype=np.int64)
feature_names = X_frame.columns.to_list()
if CONFIG["fast_max_rows"] and len(y) > CONFIG["fast_max_rows"]:
    selected, _ = train_test_split(
        np.arange(len(y)), train_size=CONFIG["fast_max_rows"], stratify=y, random_state=0
    )
    X, y = X[selected], y[selected]

rows = []
last_context = None
for repeat in range(CONFIG["repeats"]):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=repeat
    )
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)
    metrics, models = run_three_methods(
        X_train, y_train, X_test, y_test, CONFIG, "binary", repeat, DEVICE
    )
    for method, values in metrics.items():
        rows.append({"repeat": repeat, "method": method, **values})
    last_context = {
        "model": models["FedProx"], "X_train": X_train, "X_test": X_test,
        "feature_names": feature_names,
    }

run_results = pd.DataFrame(rows)
summary = run_results.groupby("method")[["accuracy", "f1", "auc"]].agg(["mean", "std"])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
run_results.to_csv(RESULTS_DIR / "kddcup2015_metrics.csv", index=False)
torch.save(last_context["model"].state_dict(), RESULTS_DIR / "kddcup2015_fedprox_last_seed.pt")
display(summary.round(4))

## 3. Compare with the paper

In [ ]:
published = pd.DataFrame(
    {
        "accuracy": [0.872, 0.876, 0.887],
        "f1": [0.922, 0.925, 0.932],
        "auc": [0.875, 0.879, 0.887],
    },
    index=["Central", "FedAvg", "FedProx"],
)
display(published.rename_axis("method").style.set_caption("Published KDD Cup 2015 results"))

## 4. Local explanations

In [ ]:
def explain_with_captum(context, sample_index=10, top_k=15):
    """Plot local LIME, Integrated Gradients, and Gradient SHAP attributions."""
    if not RUN_XAI:
        print("XAI skipped. Set AIED_RUN_XAI=1 before launching Jupyter to run this section.")
        return None
    from captum.attr import GradientShap, IntegratedGradients, Lime

    model = context["model"].to(DEVICE).eval()
    X_train = context["X_train"]
    X_test = context["X_test"]
    feature_names = np.asarray(context["feature_names"])
    sample_index = min(sample_index, len(X_test) - 1)
    sample = torch.as_tensor(X_test[[sample_index]], dtype=torch.float32, device=DEVICE)
    background = torch.as_tensor(X_train[: min(64, len(X_train))], dtype=torch.float32, device=DEVICE)

    ig_values = IntegratedGradients(model).attribute(sample, baselines=torch.zeros_like(sample), target=1)
    gs_values = GradientShap(model).attribute(sample, baselines=background, target=1, n_samples=50)
    lime_values = Lime(model).attribute(
        sample, target=1, n_samples=200, perturbations_per_eval=32,
        baselines=torch.zeros_like(sample),
    )
    values = {
        "LIME": lime_values.detach().cpu().numpy().ravel(),
        "Integrated Gradients": ig_values.detach().cpu().numpy().ravel(),
        "Gradient SHAP": gs_values.detach().cpu().numpy().ravel(),
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
    for ax, (name, attribution) in zip(axes, values.items()):
        idx = np.argsort(np.abs(attribution))[-top_k:]
        colors = np.where(attribution[idx] >= 0, "#2f6b9a", "#d18f00")
        ax.barh(feature_names[idx], attribution[idx], color=colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(name)
        ax.set_xlabel("Attribution toward dropout/positive class")
    prediction = torch.softmax(model(sample), dim=1)[0, 1].item()
    fig.suptitle(f"Sample {sample_index}: positive-class probability = {prediction:.3f}")
    plt.show()
    return values

In [ ]:
explain_with_captum(last_context, sample_index=10)